
# AI Memory Implementation

## Objective

To implement a basic AI Memory system that stores, retrieves, and reuses information from previous interactions and documents.

## Memory Types

1. Short-Term Memory
2. Long-Term Memory
3. Episodic Memory
4. Semantic Memory
5. Procedural Memory

## Implementation Scope

- Store persistent memories in a JSON file.
- Keep short-term memories in the current Python session.
- Add, search, and retrieve memories using keyword matching.
- Store conversation interactions as episodic memories.
- Extract text from documents and divide it into smaller chunks.
- Store document chunks in semantic memory.
- Verify memory persistence and document chunk storage.

## Retrieval Approach

The implementation uses keyword-based matching and ranking. It does not currently use embeddings, a vector database, or an LLM to generate answers.


### Importing Libraries and Configuring the Project

This cell imports the required Python libraries and defines the project directory, memory storage file, and supported memory types.

- **`from pathlib import Path`** – Imports `Path` for handling file and directory paths.
- **`import json`** – Imports the JSON library to read and write persistent memory data.
- **`import hashlib`** – Imports hashing functions used to generate document identifiers.
- **`from datetime import datetime, timezone`** – Imports date and time utilities for recording timestamps in UTC.
- **`from uuid import uuid4`** – Imports `uuid4()` to generate unique identifiers for memory records.
- **`PROJECT_ROOT = Path.cwd()`** – Gets the current working directory and stores it as the project root.
- **`MEMORY_FILE = PROJECT_ROOT / "ai_memory_data.json"`** – Creates a path to the JSON file used for persistent memory storage.
- **`MEMORY_TYPES = [...]`** – Defines the five supported memory categories: short-term, long-term, episodic, semantic, and procedural.
- **`Path.cwd()`** – Returns the directory from which the Python process is currently running.
- **`PROJECT_ROOT / "ai_memory_data.json"`** – Combines the project directory with the JSON filename to create a file path.
- **`print(...)`** – Displays the project directory and memory storage file path.

This setup provides the libraries and file paths used throughout the AI Memory implementation.

In [31]:
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone
from uuid import uuid4

# Use the current AI Assignment folder
PROJECT_ROOT = Path.cwd()

# Separate JSON file for AI Memory
MEMORY_FILE = PROJECT_ROOT / "ai_memory_data.json"

MEMORY_TYPES = [
    "short_term",
    "long_term",
    "episodic",
    "semantic",
    "procedural"
]

print("Project directory:", PROJECT_ROOT)
print("Memory storage file:", MEMORY_FILE)

Project directory: C:\Users\vishakha.more\AI Assignment
Memory storage file: C:\Users\vishakha.more\AI Assignment\ai_memory_data.json


### Defining Memory Types

This cell defines the five memory categories, along with a description and an example of each category.

- **`memory_type_info = {...}`** – Creates a nested dictionary containing information about each memory type.
- **`"short_term"`** – Represents temporary information and the current context.
- **`"long_term"`** – Represents useful information retained for future sessions.
- **`"episodic"`** – Represents specific events and past interactions.
- **`"semantic"`** – Represents facts, concepts, and general knowledge.
- **`"procedural"`** – Represents instructions and reusable workflows.
- **`"description"`** – Stores a description of the purpose of a memory type.
- **`"example"`** – Stores an example illustrating the memory type.
- **`memory_type_info.items()`** – Returns the dictionary's key-value pairs for iteration.
- **`for memory_type, details in memory_type_info.items()`** – Iterates through each memory category and its associated details.
- **`memory_type.upper()`** – Converts the memory type name to uppercase.
- **`replace('_', ' ')`** – Replaces underscores with spaces to make the category name easier to read.
- **`details["description"]`** – Retrieves the description associated with the current memory type.
- **`details["example"]`** – Retrieves the example associated with the current memory type.
- **`print(...)`** – Displays the memory type, its description, and its example.

This cell explains the purpose of each memory category before implementing memory storage and retrieval.

In [32]:
memory_type_info = {
    "short_term": {
        "description": "Temporarily holds recent information and current context.",
        "example": "The current conversation or task."
    },
    "long_term": {
        "description": "Retains useful information for future sessions.",
        "example": "A user's preferred programming language."
    },
    "episodic": {
        "description": "Remembers specific events and past interactions.",
        "example": "A project meeting took place on Monday."
    },
    "semantic": {
        "description": "Stores facts, concepts, and general knowledge.",
        "example": "Python is a programming language."
    },
    "procedural": {
        "description": "Stores instructions and reusable workflows.",
        "example": "Steps to process a document."
    }
}

for memory_type, details in memory_type_info.items():
    print(f"\n{memory_type.upper().replace('_', ' ')} MEMORY")
    print("Description:", details["description"])
    print("Example:", details["example"])


SHORT TERM MEMORY
Description: Temporarily holds recent information and current context.
Example: The current conversation or task.

LONG TERM MEMORY
Description: Retains useful information for future sessions.
Example: A user's preferred programming language.

EPISODIC MEMORY
Description: Remembers specific events and past interactions.
Example: A project meeting took place on Monday.

SEMANTIC MEMORY
Description: Stores facts, concepts, and general knowledge.
Example: Python is a programming language.

PROCEDURAL MEMORY
Description: Stores instructions and reusable workflows.
Example: Steps to process a document.


### Designing the Memory Storage Structure

This cell maps each memory type to its storage location and displays how temporary and persistent memories are stored.

- **`storage_design = {...}`** – Creates a dictionary mapping memory types to their storage locations.
- **`"short_term": "Python variable (temporary RAM)"`** – Specifies that short-term memory is held temporarily in a Python variable.
- **`"long_term": "ai_memory_data.json"`** – Specifies the JSON file used to store long-term memory.
- **`"episodic": "ai_memory_data.json"`** – Specifies the JSON file used to store episodic memory.
- **`"semantic": "ai_memory_data.json"`** – Specifies the JSON file used to store semantic memory.
- **`"procedural": "ai_memory_data.json"`** – Specifies the JSON file used to store procedural memory.
- **`storage_design.items()`** – Returns the memory type and storage location pairs.
- **`for memory_type, location in storage_design.items()`** – Iterates through each memory type and its assigned storage location.
- **`replace('_', ' ')`** – Replaces underscores in the memory type name with spaces.
- **`title()`** – Capitalizes the first letter of each word in the memory type name.
- **`f"{...} Memory -> {location}"`** – Formats the memory type and its storage location into a readable message.
- **`print(...)`** – Displays the storage location of each memory type.

This design separates session-based short-term memory from the four persistent memory categories stored in the JSON file.

In [33]:
storage_design = {
    "short_term": "Python variable (temporary RAM)",
    "long_term": "ai_memory_data.json",
    "episodic": "ai_memory_data.json",
    "semantic": "ai_memory_data.json",
    "procedural": "ai_memory_data.json"
}

for memory_type, location in storage_design.items():
    print(f"{memory_type.replace('_', ' ').title()} Memory -> {location}")

Short Term Memory -> Python variable (temporary RAM)
Long Term Memory -> ai_memory_data.json
Episodic Memory -> ai_memory_data.json
Semantic Memory -> ai_memory_data.json
Procedural Memory -> ai_memory_data.json


### Initializing and Managing Memory Storage

This cell defines functions to initialize, load, validate, and save persistent memory using a JSON file. It also initializes short-term memory for the current Python session.

- **`create_empty_memory()`** – Creates a dictionary containing four persistent memory types, each initialized with an empty list.
- **`"long_term": []`** – Creates an empty list to store long-term memory records.
- **`"episodic": []`** – Creates an empty list to store records of past interactions and experiences.
- **`"semantic": []`** – Creates an empty list to store factual information and knowledge.
- **`"procedural": []`** – Creates an empty list to store procedures and instructions.
- **`return {...}`** – Returns the initialized memory dictionary.
- **`load_memory()`** – Defines a function that loads persistent memories from the JSON storage file.
- **`MEMORY_FILE.exists()`** – Checks whether the memory storage file exists.
- **`if not MEMORY_FILE.exists()`** – Checks whether the storage file is missing.
- **`return create_empty_memory()`** – Returns a new empty memory structure if the file does not exist.
- **`try:`** – Starts a block for handling errors that may occur while loading the memory file.
- **`MEMORY_FILE.open("r", encoding="utf-8")`** – Opens the memory file in read mode using UTF-8 encoding.
- **`with ... as file`** – Opens the file and ensures that it is closed automatically after the block finishes.
- **`json.load(file)`** – Reads JSON data from the file and converts it into Python objects.
- **`except json.JSONDecodeError as error`** – Catches an error if the file contains invalid JSON.
- **`raise ValueError(...)`** – Raises an error with a message explaining that the memory file contains invalid JSON.
- **`from error`** – Preserves the original JSON decoding error as the cause of the new exception.
- **`isinstance(data, dict)`** – Checks whether the loaded memory data is a Python dictionary.
- **`if not isinstance(data, dict)`** – Detects whether the loaded data has an unexpected structure.
- **`expected_types = create_empty_memory()`** – Creates a reference dictionary containing the expected memory types.
- **`for memory_type in expected_types`** – Iterates through the expected memory type names.
- **`data.setdefault(memory_type, [])`** – Adds a missing memory type with an empty list without overwriting an existing value.
- **`isinstance(data[memory_type], list)`** – Checks whether the value associated with a memory type is a list.
- **`raise ValueError(...)`** – Raises an error if a memory type contains a value other than a list.
- **`return data`** – Returns the loaded and validated memory dictionary.
- **`save_memory(data)`** – Defines a function that saves the supplied memory dictionary to the JSON file.
- **`MEMORY_FILE.open("w", encoding="utf-8")`** – Opens the memory file in write mode, creating it if necessary and overwriting its existing contents.
- **`json.dump(data, file, indent=4, ensure_ascii=False)`** – Writes the memory dictionary to the JSON file. `indent=4` formats the JSON with four spaces of indentation, while `ensure_ascii=False` preserves Unicode characters.
- **`memory_store = load_memory()`** – Loads persistent memories into the `memory_store` variable.
- **`short_term_memory = []`** – Creates an empty list for short-term memory that exists only in the current Python session unless saved separately.
- **`MEMORY_FILE.exists()`** – Checks whether the memory storage file exists and returns `True` or `False`.
- **`list(memory_store.keys())`** – Retrieves the memory dictionary's keys and converts them into a list.
- **`print(...)`** – Displays the initialization status, storage file existence, and available persistent memory types.

Persistent memories can be loaded across sessions using the JSON file, while short-term memory remains in the current Python session. Changes to `memory_store` must be saved by calling `save_memory(memory_store)` to persist them.

In [38]:
def create_empty_memory():
    """Create the persistent memory structure."""
    return {
        "long_term": [],
        "episodic": [],
        "semantic": [],
        "procedural": []
    }


def load_memory():
    """Load persistent memories from the JSON file."""

    if not MEMORY_FILE.exists():
        return create_empty_memory()

    try:
        with MEMORY_FILE.open("r", encoding="utf-8") as file:
            data = json.load(file)

    except json.JSONDecodeError as error:
        raise ValueError(
            f"Invalid JSON in memory file: {MEMORY_FILE}. "
            "Please check or restore the file."
        ) from error

    if not isinstance(data, dict):
        raise ValueError("Memory data must be a JSON object.")

    expected_types = create_empty_memory()

    for memory_type in expected_types:
        data.setdefault(memory_type, [])

        if not isinstance(data[memory_type], list):
            raise ValueError(
                f"Memory type '{memory_type}' must contain a list."
            )

    return data


def save_memory(data):
    """Save persistent memories to JSON."""

    with MEMORY_FILE.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)


# Load persistent memory
memory_store = load_memory()

# Short-term memory lasts only for this Python session
short_term_memory = []

print("Memory system initialized.")
print("Storage file exists:", MEMORY_FILE.exists())
print("Persistent memory types:", list(memory_store.keys()))

Memory system initialized.
Storage file exists: True
Persistent memory types: ['long_term', 'episodic', 'semantic', 'procedural']


### Implementing Memory Addition

This cell defines a function that validates memory input, creates a memory record, prevents duplicate content within the same memory type, and stores the record in the appropriate location.

- **`add_memory(memory_type, content, metadata=None)`** – Defines a function that accepts a memory category, content, and optional metadata.
- **`memory_type not in MEMORY_TYPES`** – Checks whether the requested memory category is supported.
- **`raise ValueError(...)`** – Raises an error if an unsupported memory type is provided.
- **`isinstance(content, str)`** – Checks whether the memory content is a string.
- **`content.strip()`** – Removes leading and trailing whitespace from the content.
- **`not content.strip()`** – Checks whether the content is empty or contains only whitespace.
- **`isinstance(metadata, dict)`** – Checks whether the supplied metadata is a dictionary.
- **`metadata is not None`** – Checks whether metadata was provided.
- **`raise TypeError(...)`** – Raises an error if the supplied metadata is not a dictionary.
- **`content = content.strip()`** – Removes unnecessary whitespace from the memory content before storing it.
- **`metadata = metadata or {}`** – Uses the supplied metadata dictionary or an empty dictionary if metadata is missing or empty.
- **`existing_records = (...)`** – Selects the list in which the function should check for existing records.
- **`short_term_memory if memory_type == "short_term"`** – Selects the temporary list when the requested memory type is short-term.
- **`memory_store[memory_type]`** – Selects the corresponding persistent memory list for other memory types.
- **`for existing_record in existing_records`** – Iterates through the records already stored in the selected list.
- **`existing_record.get("content") == content`** – Checks whether an existing record has identical content.
- **`return existing_record`** – Returns the existing record instead of adding duplicate content to the same memory type.
- **`record = {...}`** – Creates a dictionary representing the new memory record.
- **`str(uuid4())`** – Generates a unique identifier and converts it to a string.
- **`"memory_type": memory_type`** – Stores the category assigned to the memory.
- **`"content": content`** – Stores the memory text.
- **`"metadata": metadata`** – Stores additional information associated with the memory.
- **`datetime.now(timezone.utc).isoformat()`** – Generates a UTC timestamp and converts it into an ISO-formatted string.
- **`if memory_type == "short_term"`** – Checks whether the new record belongs to short-term memory.
- **`short_term_memory.append(record)`** – Adds the record to the temporary in-memory list.
- **`memory_store[memory_type].append(record)`** – Adds the record to the appropriate persistent memory list.
- **`save_memory(memory_store)`** – Saves the updated persistent memory dictionary to the JSON file.
- **`return record`** – Returns the newly created or previously existing memory record.
- **`print(...)`** – Confirms that the memory storage function has been defined.

This function stores short-term memories in the current session and saves other memory types to persistent JSON storage.

In [39]:
def add_memory(memory_type, content, metadata=None):
    """Add a memory to short-term or persistent storage."""

    if memory_type not in MEMORY_TYPES:
        raise ValueError(f"Invalid memory type: {memory_type}")

    if not isinstance(content, str) or not content.strip():
        raise ValueError("Memory content must be a non-empty string.")

    if metadata is not None and not isinstance(metadata, dict):
        raise TypeError("Metadata must be a dictionary.")

    content = content.strip()
    metadata = metadata or {}

    existing_records = (
        short_term_memory
        if memory_type == "short_term"
        else memory_store[memory_type]
    )

    # Avoid adding identical content to the same memory type
    for existing_record in existing_records:
        if existing_record.get("content") == content:
            return existing_record

    record = {
        "memory_id": str(uuid4()),
        "memory_type": memory_type,
        "content": content,
        "metadata": metadata,
        "created_at": datetime.now(timezone.utc).isoformat()
    }

    if memory_type == "short_term":
        short_term_memory.append(record)
    else:
        memory_store[memory_type].append(record)
        save_memory(memory_store)

    return record


print("Memory storage function created.")

Memory storage function created.


### Adding Sample Memories

This cell adds example records to all five memory categories to demonstrate how the memory storage function works.

- **`add_memory("short_term", ...)`** – Adds a temporary memory describing the current AI Memory learning task.
- **`"I am currently learning AI Memory."`** – Provides the content of the short-term memory.
- **`{"source": "current_session"}`** – Adds metadata identifying the source as the current session.
- **`add_memory("long_term", ...)`** – Adds a persistent memory about learning Python programming.
- **`{"category": "learning_preference"}`** – Adds metadata classifying the record as a learning preference.
- **`add_memory("episodic", ...)`** – Adds a persistent record of completing a document chunking exercise.
- **`{"event": "chunking_exercise"}`** – Identifies the event associated with the episodic memory.
- **`add_memory("semantic", ...)`** – Adds a factual record about Python.
- **`{"category": "technical_fact"}`** – Classifies the semantic memory as a technical fact.
- **`add_memory("procedural", ...)`** – Adds a sequence of steps for processing a document.
- **`{"workflow": "document_processing"}`** – Identifies the associated workflow.
- **`print(...)`** – Displays a confirmation after the sample memory additions.

The sample records demonstrate how different memory types can store different kinds of information using the same function.

In [36]:
add_memory(
    "short_term",
    "I am currently learning AI Memory.",
    {"source": "current_session"}
)

add_memory(
    "long_term",
    "The user is learning Python programming.",
    {"category": "learning_preference"}
)

add_memory(
    "episodic",
    "The user completed a document chunking exercise.",
    {"event": "chunking_exercise"}
)

add_memory(
    "semantic",
    "Python is a high-level programming language.",
    {"category": "technical_fact"}
)

add_memory(
    "procedural",
    "Read a document, extract its text, split it into chunks, and save the chunks.",
    {"workflow": "document_processing"}
)

print("Sample memories added successfully.")

Sample memories added successfully.


### Displaying Stored Memories

This cell displays the contents of short-term memory and the persistent memory records loaded into the memory dictionary.

- **`print("SHORT-TERM MEMORY")`** – Displays a heading for short-term memory.
- **`"=" * 50`** – Repeats the equals sign 50 times to create a visual separator.
- **`for record in short_term_memory`** – Iterates through the short-term memory records.
- **`record["content"]`** – Retrieves the content of each short-term memory record.
- **`print("PERSISTENT MEMORY")`** – Displays a heading for persistent memory.
- **`memory_store.items()`** – Returns the memory category names and their associated record lists.
- **`for memory_type, records in memory_store.items()`** – Iterates through each persistent memory category and its records.
- **`memory_type.upper().replace('_', ' ')`** – Converts the memory category to uppercase and replaces underscores with spaces.
- **`for record in records`** – Iterates through the records in the current memory category.
- **`record["memory_id"]`** – Retrieves the unique identifier of a memory record.
- **`record["content"]`** – Retrieves the stored memory content.
- **`record["created_at"]`** – Retrieves the timestamp indicating when the record was created.
- **`print(...)`** – Displays the memory identifiers, contents, and creation timestamps.

This cell allows the stored records to be inspected and compared across memory categories.

In [37]:
print("SHORT-TERM MEMORY")
print("=" * 50)

for record in short_term_memory:
    print(record["content"])


print("\nPERSISTENT MEMORY")
print("=" * 50)

for memory_type, records in memory_store.items():
    print(f"\n{memory_type.upper().replace('_', ' ')}")

    for record in records:
        print("ID:", record["memory_id"])
        print("Content:", record["content"])
        print("Created:", record["created_at"])

SHORT-TERM MEMORY
I am currently learning AI Memory.

PERSISTENT MEMORY

LONG TERM
ID: 24de19b4-df13-41ab-9645-9fdf451d6b1d
Content: The user is learning Python programming.
Created: 2026-10-26T09:56:37.869925+00:00

EPISODIC
ID: be4efd3d-2d58-4567-9ac6-6ee5dbba35f1
Content: The user completed a document chunking exercise.
Created: 2026-10-26T09:56:37.886924+00:00
ID: 4860b4bb-917a-4489-8196-61789b089400
Content: User asked: What is semantic memory?
Assistant response: Semantic memory stores facts, concepts, and general knowledge.
Created: 2026-10-26T10:00:12.109651+00:00

SEMANTIC
ID: a670c8e0-8ae5-499a-94c1-b39ad45cca5b
Content: Python is a high-level programming language.
Created: 2026-10-26T09:56:37.913676+00:00
ID: e99165db-1467-41ef-9631-c9379f2f4c86
Content: Company Leave Policy
Employees receive 20 days of annual leave. Leave requests require manager approval.
Created: 2026-10-26T10:04:20.048072+00:00

PROCEDURAL
ID: e97c9914-aaed-47eb-b4a6-5eb90182486d
Content: Read a document

### Implementing Basic Memory Search

This cell defines a function that searches short-term and persistent memories using keyword matching.

- **`search_memory(query, memory_type=None)`** – Defines a function that accepts a search query and an optional memory category.
- **`isinstance(query, str)`** – Checks whether the query is a string.
- **`not query.strip()`** – Checks whether the query is empty or contains only whitespace.
- **`return []`** – Returns an empty list when the query is invalid or empty.
- **`memory_type is not None`** – Checks whether a specific memory category was supplied.
- **`memory_type not in MEMORY_TYPES`** – Checks whether the requested category is supported.
- **`raise ValueError(...)`** – Raises an error if the memory category is invalid.
- **`query.lower()`** – Converts the query to lowercase so that keyword matching is case-insensitive.
- **`re.findall(r"\b[a-zA-Z0-9]+\b", query.lower())`** – Extracts alphanumeric words from the query using a regular expression.
- **`set(...)`** – Converts the extracted words into a set, removing duplicate keywords.
- **`if not query_words`** – Checks whether any searchable keywords were extracted.
- **`results = []`** – Creates an empty list to store matching memory records.
- **`memory_type is None or memory_type == "short_term"`** – Determines whether short-term memories should be searched.
- **`for record in short_term_memory`** – Iterates through short-term memory records.
- **`record.get("content", "")`** – Retrieves the content of a record, using an empty string if the key is missing.
- **`re.findall(...)`** – Extracts lowercase alphanumeric keywords from the memory content.
- **`query_words.intersection(content_words)`** – Identifies keywords that appear in both the query and the memory content.
- **`if query_words.intersection(content_words)`** – Checks whether at least one keyword matches.
- **`results.append(record)`** – Adds a matching record to the results list.
- **`memory_type is None or memory_type != "short_term"`** – Determines whether persistent memories should be searched.
- **`for section, records in memory_store.items()`** – Iterates through the persistent memory categories and their records.
- **`if memory_type is not None and memory_type != section`** – Skips categories that do not match the requested memory type.
- **`return results`** – Returns the matching records.
- **`search_memory("Python programming")`** – Searches for memories that share at least one keyword with the query.
- **`len(results)`** – Calculates the number of matching records.
- **`for result in results`** – Iterates through the matching records to display their details.
- **`result["memory_type"]`** – Retrieves the category of a matching record.
- **`result["content"]`** – Retrieves the content of a matching record.

This search uses keyword overlap rather than semantic similarity. A memory is returned when at least one query keyword matches a word in its content.

In [40]:
def search_memory(query, memory_type=None):
    """Search memories using keyword matching."""

    if not isinstance(query, str) or not query.strip():
        return []

    if memory_type is not None and memory_type not in MEMORY_TYPES:
        raise ValueError(f"Invalid memory type: {memory_type}")

    query_words = set(re.findall(r"\b[a-zA-Z0-9]+\b", query.lower()))

    if not query_words:
        return []

    results = []

    # Search short-term memory
    if memory_type is None or memory_type == "short_term":
        for record in short_term_memory:
            content_words = set(
                re.findall(
                    r"\b[a-zA-Z0-9]+\b",
                    record.get("content", "").lower()
                )
            )

            if query_words.intersection(content_words):
                results.append(record)

    # Search persistent memory
    if memory_type is None or memory_type != "short_term":
        for section, records in memory_store.items():
            if memory_type is not None and memory_type != section:
                continue

            for record in records:
                content_words = set(
                    re.findall(
                        r"\b[a-zA-Z0-9]+\b",
                        record.get("content", "").lower()
                    )
                )

                if query_words.intersection(content_words):
                    results.append(record)

    return results


# Test basic search
results = search_memory("Python programming")

print("Search results:", len(results))

for result in results:
    print("\nType:", result["memory_type"])
    print("Content:", result["content"])

Search results: 2

Type: long_term
Content: The user is learning Python programming.

Type: semantic
Content: Python is a high-level programming language.


### Inspecting Persistent Memory Data

This cell checks whether the JSON storage file exists and displays its saved contents.

- **`MEMORY_FILE.exists()`** – Checks whether the persistent memory file exists.
- **`if MEMORY_FILE.exists()`** – Executes the file-reading block only when the file exists.
- **`MEMORY_FILE.open("r", encoding="utf-8")`** – Opens the file in read mode using UTF-8 encoding.
- **`with ... as file`** – Ensures that the file is closed automatically after reading.
- **`json.load(file)`** – Converts the JSON contents into Python objects and stores them in `saved_data`.
- **`json.dumps(saved_data, indent=4, ensure_ascii=False)`** – Converts the loaded data back into a formatted JSON string for display.
- **`indent=4`** – Adds four-space indentation to make the JSON easier to read.
- **`ensure_ascii=False`** – Preserves Unicode characters in the displayed JSON.
- **`print(...)`** – Displays the complete saved memory structure.
- **`else`** – Executes the alternative block when the storage file does not exist.
- **`print("No persistent memories have been saved yet.")`** – Displays a message when no storage file is available.

This cell helps inspect the actual JSON records saved by the memory system.

In [11]:
if MEMORY_FILE.exists():
    with MEMORY_FILE.open("r", encoding="utf-8") as file:
        saved_data = json.load(file)

    print(json.dumps(saved_data, indent=4, ensure_ascii=False))
else:
    print("No persistent memories have been saved yet.")

{
    "long_term": [
        {
            "memory_id": "24de19b4-df13-41ab-9645-9fdf451d6b1d",
            "memory_type": "long_term",
            "content": "The user is learning Python programming.",
            "metadata": {
                "category": "learning_preference"
            },
            "created_at": "2026-10-26T09:56:37.869925+00:00"
        }
    ],
    "episodic": [
        {
            "memory_id": "be4efd3d-2d58-4567-9ac6-6ee5dbba35f1",
            "memory_type": "episodic",
            "content": "The user completed a document chunking exercise.",
            "metadata": {
                "event": "chunking_exercise"
            },
            "created_at": "2026-10-26T09:56:37.886924+00:00"
        }
    ],
    "semantic": [
        {
            "memory_id": "a670c8e0-8ae5-499a-94c1-b39ad45cca5b",
            "memory_type": "semantic",
            "content": "Python is a high-level programming language.",
            "metadata": {
                "category": 

### Verifying Persistent Memory

This cell reloads the saved memory data, displays the number of records in each category, and checks whether each record's memory type matches its category.

- **`reloaded_memory = load_memory()`** – Loads the latest persistent memory data from the JSON file.
- **`print("PERSISTENCE VERIFICATION")`** – Displays the verification heading.
- **`reloaded_memory.items()`** – Returns the memory category names and their associated record lists.
- **`for memory_type, records in reloaded_memory.items()`** – Iterates through the memory categories and their records.
- **`len(records)`** – Counts the records stored in each category.
- **`record["memory_type"] == memory_type`** – Checks whether a record's declared memory type matches the category containing it.
- **`all(...)`** – Returns `True` only if every comparison in the generator expression is true.
- **`assert ...`** – Raises an `AssertionError` if any record has a mismatched memory type.
- **`print(...)`** – Displays the verification result if the assertion passes.

This check verifies category consistency after reloading. It does not, by itself, prove that every memory record matches the data originally saved.

In [12]:
reloaded_memory = load_memory()

print("PERSISTENCE VERIFICATION")
print("=" * 50)

for memory_type, records in reloaded_memory.items():
    print(f"{memory_type}: {len(records)} record(s)")

assert all(
    record["memory_type"] == memory_type
    for memory_type, records in reloaded_memory.items()
    for record in records
)

print("\nPersistent memory successfully reloaded.")

PERSISTENCE VERIFICATION
long_term: 1 record(s)
episodic: 1 record(s)
semantic: 1 record(s)
procedural: 1 record(s)

Persistent memory successfully reloaded.


### Displaying the Memory Storage Summary

This cell displays the number of short-term and persistent memory records and identifies the JSON storage file.

- **`print("MEMORY STORAGE SUMMARY")`** – Displays the summary heading.
- **`len(short_term_memory)`** – Counts the short-term records currently held in the Python session.
- **`reloaded_memory.items()`** – Provides the persistent memory categories and their record lists.
- **`memory_type.replace('_', ' ').title()`** – Converts the memory category name into a readable title.
- **`len(records)`** – Counts the records in the current persistent memory category.
- **`MEMORY_FILE`** – Refers to the path of the persistent JSON storage file.
- **`print(...)`** – Displays the record counts and storage file path.

This summary provides a quick overview of the current memory storage state.

In [13]:
print("MEMORY STORAGE SUMMARY")
print("=" * 50)

print(
    "Short-term memory:",
    len(short_term_memory),
    "record(s) in current Python session"
)

for memory_type, records in reloaded_memory.items():
    print(
        f"{memory_type.replace('_', ' ').title()} memory:",
        len(records),
        "record(s) in JSON storage"
    )

print("\nJSON file:", MEMORY_FILE)

MEMORY STORAGE SUMMARY
Short-term memory: 1 record(s) in current Python session
Long Term memory: 1 record(s) in JSON storage
Episodic memory: 1 record(s) in JSON storage
Semantic memory: 1 record(s) in JSON storage
Procedural memory: 1 record(s) in JSON storage

JSON file: C:\Users\vishakha.more\AI Assignment\ai_memory_data.json


### Resetting the Memory Demonstration

This cell provides an optional way to clear the memory records created by the notebook demonstration.

- **`confirm_reset = False`** – Disables the reset operation by default.
- **`if confirm_reset`** – Checks whether the reset has been explicitly enabled.
- **`memory_store = create_empty_memory()`** – Replaces the in-memory persistent memory dictionary with an empty structure.
- **`short_term_memory.clear()`** – Removes all records from the existing short-term memory list.
- **`save_memory(memory_store)`** – Writes the empty persistent memory structure to the JSON file, replacing its existing contents.
- **`print("Memory demonstration reset.")`** – Displays a confirmation after the reset executes.
- **`else`** – Executes when the reset has not been enabled.
- **`print("Nothing deleted...")`** – Confirms that the reset was skipped and explains how to enable it.

**Caution:** Setting `confirm_reset = True` deletes all persistent memory records in the configured JSON file, including records that may have existed before the demonstration. Use this option only when you intend to reset that file.

In [14]:
# CAUTION: This deletes all records in this notebook's JSON memory file.
confirm_reset = False

if confirm_reset:
    memory_store = create_empty_memory()
    short_term_memory.clear()
    save_memory(memory_store)
    print("Memory demonstration reset.")
else:
    print("Nothing deleted. Set confirm_reset = True to reset.")

Nothing deleted. Set confirm_reset = True to reset.


### Implementing Keyword-Based Memory Retrieval

This cell defines a stop-word list, converts text into searchable keywords, and retrieves memories ranked by keyword overlap.

- **`STOP_WORDS = {...}`** – Defines a set of common words that are excluded from keyword matching.
- **`def tokenize(text)`** – Defines a function that converts text into a normalized set of keywords.
- **`text.lower()`** – Converts the input text to lowercase for case-insensitive matching.
- **`re.findall(r"\b[a-zA-Z0-9]+\b", text.lower())`** – Extracts alphanumeric words from the text.
- **`word for word in words if word not in STOP_WORDS`** – Keeps words that are not in the stop-word set.
- **`set(...)`** – Removes duplicate keywords from the tokenized text.
- **`retrieve_memories(query, top_k=5, memory_type=None, min_score=0.25)`** – Defines a function that retrieves and ranks memories using keyword overlap.
- **`top_k=5`** – Sets the default maximum number of results to return.
- **`memory_type=None`** – Allows searching all memory categories unless a specific category is supplied.
- **`min_score=0.25`** – Sets the minimum keyword-overlap score required for a result to be included.
- **`isinstance(query, str)`** – Checks whether the query is a string.
- **`not query.strip()`** – Detects an empty or whitespace-only query.
- **`return []`** – Returns no results for an invalid or empty query.
- **`isinstance(top_k, int)`** – Checks whether the requested result limit is an integer.
- **`top_k <= 0`** – Checks whether the requested result limit is zero or negative.
- **`raise ValueError(...)`** – Raises an error if `top_k` is not a positive integer.
- **`memory_type not in MEMORY_TYPES`** – Checks whether the specified memory category is supported.
- **`0 <= min_score <= 1`** – Checks whether the minimum score lies between zero and one, inclusive.
- **`query_words = tokenize(query)`** – Converts the query into a set of searchable keywords.
- **`candidates = []`** – Creates a list to collect candidate memory records.
- **`candidates.extend(short_term_memory)`** – Adds short-term records to the candidates when the selected category allows them.
- **`persistent_data = load_memory()`** – Reloads persistent memory from the JSON file so saved records are included.
- **`for section, records in persistent_data.items()`** – Iterates through persistent memory categories and their records.
- **`memory_type is None or memory_type == section`** – Includes records from all categories or only the selected category.
- **`candidates.extend(records)`** – Adds the selected persistent records to the candidate list.
- **`results = []`** – Creates a list for memories that pass the retrieval checks.
- **`record.get("content", "")`** – Retrieves the memory content, using an empty string if the key is missing.
- **`content_words = tokenize(content)`** – Converts the memory content into searchable keywords.
- **`query_words.intersection(content_words)`** – Finds keywords shared by the query and memory content.
- **`if not matched_words`** – Skips records with no matching keywords.
- **`len(matched_words) / len(query_words)`** – Calculates the score as the proportion of query keywords found in the memory.
- **`if score < min_score`** – Skips records whose scores are below the minimum threshold.
- **`round(score, 2)`** – Rounds the score to two decimal places for display.
- **`sorted(matched_words)`** – Produces an alphabetically ordered list of matched keywords.
- **`results.append({...})`** – Stores the relevant memory details, score, and matched keywords in the results list.
- **`results.sort(key=lambda item: item["score"], reverse=True)`** – Sorts the results from highest score to lowest score.
- **`return results[:top_k]`** – Returns at most the requested number of highest-ranked results.

This retrieval method uses keyword overlap to rank memories. Its score measures the proportion of query keywords matched, not the deeper semantic similarity of the text.

In [41]:
STOP_WORDS = {
    "a", "an", "the", "is", "are", "was", "were",
    "am", "i", "me", "my", "you", "your", "we",
    "our", "it", "this", "that", "what", "which",
    "how", "do", "does", "did", "to", "of", "in",
    "on", "for", "and", "or", "with", "about",
    "from", "can", "could", "would", "should"
}


def tokenize(text):
    """Convert text into normalized keywords."""
    words = re.findall(r"\b[a-zA-Z0-9]+\b", text.lower())
    return {word for word in words if word not in STOP_WORDS}


def retrieve_memories(
    query,
    top_k=5,
    memory_type=None,
    min_score=0.25
):
    """Retrieve and rank memories using keyword overlap."""

    if not isinstance(query, str) or not query.strip():
        return []

    if not isinstance(top_k, int) or top_k <= 0:
        raise ValueError("top_k must be a positive integer.")

    if memory_type is not None and memory_type not in MEMORY_TYPES:
        raise ValueError(f"Invalid memory type: {memory_type}")

    if not 0 <= min_score <= 1:
        raise ValueError("min_score must be between 0 and 1.")

    query_words = tokenize(query)

    if not query_words:
        return []

    candidates = []

    # Include short-term memory when requested
    if memory_type is None or memory_type == "short_term":
        candidates.extend(short_term_memory)

    # Reload persistent memories so saved data is included
    persistent_data = load_memory()

    for section, records in persistent_data.items():
        if memory_type is None or memory_type == section:
            candidates.extend(records)

    results = []

    for record in candidates:
        content = record.get("content", "")
        content_words = tokenize(content)

        matched_words = query_words.intersection(content_words)

        if not matched_words:
            continue

        score = len(matched_words) / len(query_words)

        if score < min_score:
            continue

        results.append({
            "memory_id": record.get("memory_id"),
            "memory_type": record.get("memory_type"),
            "content": content,
            "metadata": record.get("metadata", {}),
            "score": round(score, 2),
            "matched_words": sorted(matched_words)
        })

    results.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return results[:top_k]

### Testing Ranked Memory Retrieval

This cell searches stored memories for a query and displays the retrieved records with their scores and matched keywords.

- **`query = "Python programming"`** – Defines the text used to search for relevant memories.
- **`retrieve_memories(query, top_k=5)`** – Retrieves up to five memories ranked by keyword overlap.
- **`print("Query:", query)`** – Displays the search query.
- **`if results`** – Checks whether any matching memories were retrieved.
- **`enumerate(results, start=1)`** – Iterates through the results while assigning numbers starting from one.
- **`result["memory_type"]`** – Retrieves the memory category of each result.
- **`result["content"]`** – Retrieves the content of each memory.
- **`result["score"]`** – Retrieves the calculated keyword-overlap score.
- **`result["matched_words"]`** – Retrieves the keywords shared by the query and the memory content.
- **`else`** – Handles the case where no memories match the query.
- **`print(...)`** – Displays the retrieved information or a message indicating that no matching memories were found.

This test demonstrates how the retrieval function ranks and presents memories relevant to a search query.

In [16]:
query = "Python programming"

results = retrieve_memories(query, top_k=5)

print("Query:", query)
print("=" * 60)

if results:
    for index, result in enumerate(results, start=1):
        print(f"\nResult {index}")
        print("Memory type:", result["memory_type"])
        print("Content:", result["content"])
        print("Score:", result["score"])
        print("Matched words:", result["matched_words"])
else:
    print("No matching memories found.")

Query: Python programming

Result 1
Memory type: long_term
Content: The user is learning Python programming.
Score: 1.0
Matched words: ['programming', 'python']

Result 2
Memory type: semantic
Content: Python is a high-level programming language.
Score: 1.0
Matched words: ['programming', 'python']


### Testing Multiple Memory Retrieval Queries

This cell tests the retrieval function with multiple queries to examine how different search terms retrieve different memories.

- **`test_queries = [...]`** – Defines a list of sample queries covering programming, document chunks, the current AI Memory task, and document processing.
- **`for query in test_queries`** – Iterates through each query in the list.
- **`retrieve_memories(query, top_k=3)`** – Retrieves up to three matching memories for the current query.
- **`print(f"\nQuery: {query}")`** – Displays each query on a new line.
- **`for result in results`** – Iterates through the retrieved results.
- **`result['memory_type']`** – Retrieves the category of each matching memory.
- **`result['content']`** – Retrieves the memory content.
- **`result['score']`** – Retrieves the keyword-overlap score.
- **`f"...{...}..."`** – Uses an f-string to combine the memory category, content, and score into a readable output.
- **`print(...)`** – Displays the results for each query.

This test helps evaluate the retrieval behavior across several queries. Results depend on keyword overlap, so related wording without shared keywords may not be retrieved.

In [17]:
test_queries = [
    "Python programming",
    "document chunks",
    "current AI Memory task",
    "document processing workflow"
]

for query in test_queries:
    results = retrieve_memories(query, top_k=3)

    print(f"\nQuery: {query}")
    for result in results:
        print(
            f"- [{result['memory_type']}] "
            f"{result['content']} "
            f"(score={result['score']})"
        )


Query: Python programming
- [long_term] The user is learning Python programming. (score=1.0)
- [semantic] Python is a high-level programming language. (score=1.0)

Query: document chunks
- [procedural] Read a document, extract its text, split it into chunks, and save the chunks. (score=1.0)
- [episodic] The user completed a document chunking exercise. (score=0.5)

Query: current AI Memory task
- [short_term] I am currently learning AI Memory. (score=0.5)

Query: document processing workflow
- [episodic] The user completed a document chunking exercise. (score=0.33)
- [procedural] Read a document, extract its text, split it into chunks, and save the chunks. (score=0.33)


### Generating Responses from Retrieved Memories

This cell defines a function that collects retrieved memory records and combines their contents into a response.

- **`answer_from_memory(question, top_k=3)`** – Defines a function that accepts a question and the maximum number of memories to retrieve.
- **`retrieve_memories(question, top_k=top_k)`** – Retrieves the highest-ranked memories matching the question.
- **`if not results`** – Checks whether the retrieval function returned any results.
- **`return {"answer": ..., "sources": []}`** – Returns a message indicating that no relevant information was found and an empty source list.
- **`answer_parts = [...]`** – Creates a list of formatted strings containing the category and content of each retrieved memory.
- **`f"[{result['memory_type']}] {result['content']}"`** – Formats each memory with its category and content.
- **`"\n".join(answer_parts)`** – Combines the retrieved memory strings into one response, separating them with newline characters.
- **`"sources": results`** – Includes the retrieved records so that their contents and scores can be inspected.
- **`response = answer_from_memory(...)`** – Calls the function with a sample question about Python.
- **`response["answer"]`** – Retrieves the combined response text.
- **`response["sources"]`** – Retrieves the memory records used to construct the response.
- **`for source in response["sources"]`** – Iterates through the retrieved source records.
- **`source['memory_type']`** – Retrieves the category of each source.
- **`source['score']`** – Retrieves the relevance score calculated by the keyword retrieval function.
- **`source['content']`** – Retrieves the content of each source memory.
- **`print(...)`** – Displays the combined response and the retrieved memory sources.

The function assembles a response from stored information. It does not use an LLM to generate new answers; the response consists of retrieved memory content.

In [42]:
def answer_from_memory(question, top_k=3):
    """Return information retrieved from stored memories."""

    results = retrieve_memories(question, top_k=top_k)

    if not results:
        return {
            "answer": "No relevant information was found in memory.",
            "sources": []
        }

    answer_parts = [
        f"[{result['memory_type']}] {result['content']}"
        for result in results
    ]

    return {
        "answer": "\n".join(answer_parts),
        "sources": results
    }


response = answer_from_memory("What do I know about Python?")

print("RETRIEVED INFORMATION")
print(response["answer"])

print("\nMEMORY SOURCES")

for source in response["sources"]:
    print(
        f"{source['memory_type']} | "
        f"Score: {source['score']} | "
        f"Content: {source['content']}"
    )

RETRIEVED INFORMATION
[long_term] The user is learning Python programming.
[semantic] Python is a high-level programming language.

MEMORY SOURCES
long_term | Score: 0.5 | Content: The user is learning Python programming.
semantic | Score: 0.5 | Content: Python is a high-level programming language.


### Storing Conversations as Episodic Memories

This cell defines a function that saves a question and its corresponding answer as a persistent episodic memory.

- **`remember_interaction(question, answer)`** – Defines a function that accepts a user question and an assistant response.
- **`add_memory("episodic", ...)`** – Stores the interaction in the episodic memory category.
- **`f"User asked: {question}\nAssistant response: {answer}"`** – Combines the question and answer into one text record, separated by a newline.
- **`metadata={...}`** – Supplies additional information about the stored interaction.
- **`"event_type": "conversation"`** – Labels the record as a conversation event.
- **`"question": question`** – Stores the original question in the metadata.
- **`return record`** – Returns the memory record created by `add_memory()`.
- **`question = "What is semantic memory?"`** – Defines a sample user question.
- **`answer = (...)`** – Defines a sample answer describing semantic memory.
- **`remember_interaction(question, answer)`** – Saves the sample interaction and stores the returned record in `saved_interaction`.
- **`saved_interaction["memory_type"]`** – Retrieves the category assigned to the saved record.
- **`saved_interaction["memory_id"]`** – Retrieves the unique identifier of the saved record.
- **`saved_interaction["content"]`** – Retrieves the combined question-and-answer text.
- **`print(...)`** – Displays a confirmation and details of the saved interaction.

This function demonstrates how previous conversations can be recorded as episodic memories for later retrieval.

In [19]:
def remember_interaction(question, answer):
    record = add_memory(
        "episodic",
        f"User asked: {question}\nAssistant response: {answer}",
        metadata={
            "event_type": "conversation",
            "question": question
        }
    )

    return record


question = "What is semantic memory?"
answer = (
    "Semantic memory stores facts, concepts, and general knowledge."
)

saved_interaction = remember_interaction(question, answer)

print("Conversation saved.")
print("Memory type:", saved_interaction["memory_type"])
print("Memory ID:", saved_interaction["memory_id"])
print("Content:", saved_interaction["content"])

Conversation saved.
Memory type: episodic
Memory ID: 4860b4bb-917a-4489-8196-61789b089400
Content: User asked: What is semantic memory?
Assistant response: Semantic memory stores facts, concepts, and general knowledge.


### Verifying Episodic Memory Storage

This cell reloads persistent memory and displays the three most recently stored episodic records.

- **`reloaded_memory = load_memory()`** – Loads the latest persistent memory data from the JSON file.
- **`reloaded_memory["episodic"]`** – Retrieves the list of episodic memory records.
- **`len(episodic_records)`** – Calculates the total number of episodic records.
- **`episodic_records[-3:]`** – Selects the last three records in the list, or all available records if fewer than three exist.
- **`for record in episodic_records[-3:]`** – Iterates through the selected records.
- **`record["memory_id"]`** – Retrieves the unique identifier of each record.
- **`record["content"]`** – Retrieves the stored interaction text.
- **`print(...)`** – Displays the total record count and the details of the selected records.

This cell helps verify that episodic interactions have been saved and can be retrieved from persistent storage.

In [20]:
reloaded_memory = load_memory()

episodic_records = reloaded_memory["episodic"]

print("Total episodic memories:", len(episodic_records))

for record in episodic_records[-3:]:
    print("\nMemory ID:", record["memory_id"])
    print("Content:", record["content"])

Total episodic memories: 2

Memory ID: be4efd3d-2d58-4567-9ac6-6ee5dbba35f1
Content: The user completed a document chunking exercise.

Memory ID: 4860b4bb-917a-4489-8196-61789b089400
Content: User asked: What is semantic memory?
Assistant response: Semantic memory stores facts, concepts, and general knowledge.


### Displaying the AI Memory System Status

This cell displays the current storage file, the short-term memory count, and the number of persistent records in each memory category.

- **`print("AI MEMORY SYSTEM STATUS")`** – Displays the system status heading.
- **`MEMORY_FILE`** – Identifies the path of the persistent memory file.
- **`MEMORY_FILE.exists()`** – Checks whether the storage file exists.
- **`len(short_term_memory)`** – Counts the short-term records in the current Python session.
- **`persistent_data = load_memory()`** – Reloads the latest persistent memory data.
- **`persistent_data.items()`** – Returns the persistent memory category names and their record lists.
- **`for memory_type, records in persistent_data.items()`** – Iterates through all persistent memory categories.
- **`len(records)`** – Counts the records in each category.
- **`print(...)`** – Displays the storage file, file existence, short-term record count, and persistent memory counts.

This status report provides an overview of the memory system's current storage state.

In [21]:
print("AI MEMORY SYSTEM STATUS")
print("=" * 50)

print("Storage file:", MEMORY_FILE)
print("File exists:", MEMORY_FILE.exists())

print("\nShort-term memory:")
print("Records:", len(short_term_memory))

persistent_data = load_memory()

print("\nPersistent memory:")
for memory_type, records in persistent_data.items():
    print(f"{memory_type}: {len(records)} record(s)")

AI MEMORY SYSTEM STATUS
Storage file: C:\Users\vishakha.more\AI Assignment\ai_memory_data.json
File exists: True

Short-term memory:
Records: 1

Persistent memory:
long_term: 1 record(s)
episodic: 2 record(s)
semantic: 1 record(s)
procedural: 1 record(s)


### Selecting the Document for Processing

This cell defines the path of the input text document and checks whether the file is available for processing.

- **`DOCUMENT_PATH = PROJECT_ROOT / "sample.txt"`** – Creates the path to `sample.txt` inside the project directory.
- **`DOCUMENT_PATH.exists()`** – Checks whether the selected document exists.
- **`if not DOCUMENT_PATH.exists()`** – Detects when the document is missing.
- **`raise FileNotFoundError(...)`** – Stops execution with an error message if the document cannot be found.
- **`f"Document not found: {DOCUMENT_PATH}\n"`** – Includes the expected document path in the error message.
- **`"Place sample.txt in the project folder or update DOCUMENT_PATH."`** – Explains how to resolve the missing-file issue.
- **`print("Selected document:", DOCUMENT_PATH)`** – Displays the selected document path.
- **`print("Document exists:", DOCUMENT_PATH.exists())`** – Displays whether the file exists.

The document must be available at the specified path before text extraction can proceed.

In [43]:
DOCUMENT_PATH = PROJECT_ROOT / "sample.txt"

if not DOCUMENT_PATH.exists():
    raise FileNotFoundError(
        f"Document not found: {DOCUMENT_PATH}\n"
        "Place sample.txt in the project folder or update DOCUMENT_PATH."
    )

print("Selected document:", DOCUMENT_PATH)
print("Document exists:", DOCUMENT_PATH.exists())

Selected document: C:\Users\vishakha.more\AI Assignment\sample.txt
Document exists: True


### Extracting Text from a Document

This cell defines a function that validates a document path, checks the file extension, reads the text, and rejects empty documents.

- **`def extract_text(file_path)`** – Defines a function that accepts the path of a document.
- **`Path(file_path)`** – Converts the supplied path into a `Path` object.
- **`file_path.exists()`** – Checks whether the input file exists.
- **`raise FileNotFoundError(file_path)`** – Raises an error if the document does not exist.
- **`file_path.suffix.lower()`** – Retrieves the file extension and converts it to lowercase.
- **`!= ".txt"`** – Checks whether the extension is different from `.txt`.
- **`raise ValueError(...)`** – Rejects files that do not use the supported `.txt` extension.
- **`file_path.read_text(encoding="utf-8-sig")`** – Reads the file as text and handles a UTF-8 byte-order mark if present.
- **`strip()`** – Removes leading and trailing whitespace from the extracted text.
- **`if not text`** – Checks whether the document contains no usable text after trimming whitespace.
- **`raise ValueError("The document is empty.")`** – Raises an error if the extracted text is empty.
- **`return text`** – Returns the extracted document text.
- **`document_text = extract_text(DOCUMENT_PATH)`** – Calls the function and stores the extracted text.
- **`DOCUMENT_PATH.name`** – Retrieves the document filename without its parent directory.
- **`len(document_text)`** – Calculates the number of characters in the extracted text.
- **`document_text[:1000]`** – Selects the first 1,000 characters as a preview.
- **`print(...)`** – Displays the extraction status, filename, character count, and text preview.

This implementation currently supports plain-text `.txt` documents only; it does not extract text directly from PDF or DOCX files.

In [23]:
def extract_text(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(file_path)

    if file_path.suffix.lower() != ".txt":
        raise ValueError(
            "This cell currently supports .txt files only."
        )

    text = file_path.read_text(encoding="utf-8-sig").strip()

    if not text:
        raise ValueError("The document is empty.")

    return text


document_text = extract_text(DOCUMENT_PATH)

print("Extraction successful.")
print("Filename:", DOCUMENT_PATH.name)
print("Character count:", len(document_text))
print("\nExtracted text preview:\n")
print(document_text[:1000])

Extraction successful.
Filename: sample.txt
Character count: 104

Extracted text preview:

Company Leave Policy
Employees receive 20 days of annual leave.
Leave requests require manager approval.


### Implementing Sentence-Aware Document Chunking

This cell defines a function that splits document text into chunks with a maximum character limit and optional overlap between chunks.

- **`split_into_chunks(text, max_chars=500, overlap=0)`** – Defines a function with a default maximum chunk size of 500 characters and no overlap.
- **`max_chars <= 0`** – Checks whether the maximum chunk size is zero or negative.
- **`raise ValueError(...)`** – Rejects an invalid maximum chunk size.
- **`overlap < 0 or overlap >= max_chars`** – Checks whether the overlap is negative or greater than or equal to the maximum chunk size.
- **`raise ValueError(...)`** – Rejects an invalid overlap configuration.
- **`re.split(r"(?<=[.!?])\s+", text.strip())`** – Splits the trimmed text at whitespace following sentence-ending punctuation such as periods, exclamation marks, and question marks.
- **`chunks = []`** – Creates a list to store the generated chunks.
- **`current_chunk = ""`** – Initializes an empty string for building the current chunk.
- **`for sentence in sentences`** – Iterates through the extracted sentences.
- **`sentence.strip()`** – Removes leading and trailing whitespace from each sentence.
- **`if not sentence`** – Skips empty sentences.
- **`len(sentence) > max_chars`** – Checks whether a sentence exceeds the maximum chunk size.
- **`chunks.append(current_chunk)`** – Saves the current chunk before processing an oversized sentence.
- **`step = max_chars - overlap`** – Calculates the distance between the starting positions of consecutive pieces of an oversized sentence.
- **`range(0, len(sentence), step)`** – Generates the starting positions used to split an oversized sentence into pieces.
- **`sentence[start:start + max_chars]`** – Extracts a piece of the sentence with at most `max_chars` characters.
- **`if piece`** – Checks whether the extracted piece is non-empty.
- **`chunks.append(piece)`** – Adds the piece to the chunk list.
- **`continue`** – Skips the remaining loop statements and proceeds to the next sentence.
- **`candidate = (...)`** – Builds a candidate chunk by combining the current chunk and the next sentence when a current chunk exists.
- **`f"{current_chunk} {sentence}".strip()`** – Combines the current chunk and sentence with a space, then removes leading and trailing whitespace.
- **`len(candidate) <= max_chars`** – Checks whether the combined text fits within the maximum chunk size.
- **`current_chunk = candidate`** – Keeps the combined text as the current chunk when it fits.
- **`chunks.append(current_chunk)`** – Saves the current chunk when the next sentence cannot fit.
- **`if overlap`** – Checks whether overlap is enabled.
- **`current_chunk[-overlap:]`** – Extracts the final `overlap` characters from the current chunk.
- **`current_chunk = f"{current_chunk} {sentence}".strip()`** – Combines the retained overlap with the next sentence.
- **`if len(current_chunk) > max_chars`** – Checks whether the combined text exceeds the maximum chunk size.
- **`current_chunk = sentence`** – Uses the sentence alone if the overlap-based combination exceeds the limit.
- **`if current_chunk`** – Checks whether any text remains in the final chunk.
- **`chunks.append(current_chunk)`** – Adds the final non-empty chunk to the list.
- **`return chunks`** – Returns the generated chunks.
- **`split_into_chunks(document_text, max_chars=500, overlap=50)`** – Splits the extracted document into chunks with a 500-character limit and a configured overlap of 50 characters.
- **`enumerate(chunks, start=1)`** – Iterates through the chunks and numbers them starting from one.
- **`len(chunk)`** – Calculates the number of characters in each chunk.
- **`print(...)`** – Displays the document filename, total number of chunks, and each chunk's content and length.

Sentence-aware chunking attempts to keep sentences together when possible. Sentences longer than the size limit are split into smaller pieces. The configured overlap is handled differently for oversized sentences and normal sentence boundaries.

In [44]:
def split_into_chunks(text, max_chars=500, overlap=0):
    """Split text into sentence-aware chunks with optional overlap."""

    if max_chars <= 0:
        raise ValueError("max_chars must be greater than zero.")

    if overlap < 0 or overlap >= max_chars:
        raise ValueError(
            "overlap must be at least 0 and less than max_chars."
        )

    sentences = re.split(r"(?<=[.!?])\s+", text.strip())

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        sentence = sentence.strip()

        if not sentence:
            continue

        # Split sentences that exceed the chunk limit
        if len(sentence) > max_chars:
            if current_chunk:
                chunks.append(current_chunk)
                current_chunk = ""

            step = max_chars - overlap

            for start in range(0, len(sentence), step):
                piece = sentence[start:start + max_chars]

                if piece:
                    chunks.append(piece)

            continue

        candidate = (
            f"{current_chunk} {sentence}".strip()
            if current_chunk else sentence
        )

        if len(candidate) <= max_chars:
            current_chunk = candidate
        else:
            chunks.append(current_chunk)

            if overlap:
                current_chunk = current_chunk[-overlap:].strip()
                current_chunk = f"{current_chunk} {sentence}".strip()
            else:
                current_chunk = sentence

            # Handle a sentence that does not fit after adding overlap
            if len(current_chunk) > max_chars:
                current_chunk = sentence

    if current_chunk:
        chunks.append(current_chunk)

    return chunks


chunks = split_into_chunks(
    document_text,
    max_chars=500,
    overlap=50
)

print("Document:", DOCUMENT_PATH.name)
print("Total chunks:", len(chunks))

for index, chunk in enumerate(chunks, start=1):
    print(f"\nCHUNK {index} | {len(chunk)} characters")
    print(chunk)

Document: sample.txt
Total chunks: 1

CHUNK 1 | 104 characters
Company Leave Policy
Employees receive 20 days of annual leave. Leave requests require manager approval.


### Storing Document Chunks in Persistent Semantic Memory

This cell generates a document identifier, removes older chunks associated with the same document, stores the current chunks as semantic memories, and saves the updated memory data.

- **`document_text.encode("utf-8")`** – Converts the extracted document text into bytes for hashing.
- **`hashlib.sha256(...)`** – Computes a SHA-256 hash of the document content.
- **`.hexdigest()`** – Converts the hash into a hexadecimal string.
- **`document_content_hash`** – Stores the hash calculated from the extracted document content.
- **`f"{DOCUMENT_PATH.name}:{document_content_hash}"`** – Combines the filename and content hash into one string.
- **`hashlib.sha256(...).hexdigest()`** – Generates a stable identifier from the filename and content hash.
- **`document_id`** – Identifies the document using its filename and content.
- **`datetime.now(timezone.utc).isoformat()`** – Generates a UTC timestamp in ISO format.
- **`memory_store = load_memory()`** – Loads the latest persistent memory data before modifying it.
- **`memory_store["semantic"]`** – Accesses the list of semantic memory records.
- **`for record in memory_store["semantic"]`** – Iterates through the existing semantic records.
- **`record.get("metadata", {})`** – Retrieves a record's metadata, using an empty dictionary if it is missing.
- **`.get("document_id") != document_id`** – Keeps records whose document identifier does not match the current document.
- **`memory_store["semantic"] = [...]`** – Replaces the semantic memory list with the filtered records, removing older chunks belonging to this same document identifier.
- **`enumerate(chunks)`** – Iterates through the chunks and assigns zero-based indexes.
- **`record = {...}`** – Creates a memory record for each document chunk.
- **`str(uuid4())`** – Generates a unique identifier for each chunk record.
- **`"memory_type": "semantic"`** – Classifies each chunk as a semantic memory.
- **`"content": chunk`** – Stores the chunk text as the memory content.
- **`"document_id": document_id`** – Associates the chunk with its source document.
- **`"filename": DOCUMENT_PATH.name`** – Stores the source document's filename in the metadata.
- **`"chunk_index": index`** – Stores the chunk's zero-based position in the document.
- **`"source_type": "document"`** – Identifies the record as originating from a document.
- **`"stored_at": stored_at`** – Records the storage timestamp in the metadata.
- **`"created_at": stored_at`** – Stores the same timestamp as the record creation time.
- **`memory_store["semantic"].append(record)`** – Adds the chunk record to semantic memory.
- **`save_memory(memory_store)`** – Writes the updated persistent memory dictionary to the JSON file.
- **`print(...)`** – Displays the storage confirmation, document identifier, number of chunks saved, and storage file path.

This process stores document chunks as persistent semantic memory records and associates each chunk with its source document and position.

In [52]:
# Generate a stable ID for this document
document_content_hash = hashlib.sha256(
    document_text.encode("utf-8")
).hexdigest()

document_id = hashlib.sha256(
    f"{DOCUMENT_PATH.name}:{document_content_hash}".encode("utf-8")
).hexdigest()

stored_at = datetime.now(timezone.utc).isoformat()

# Load the latest persistent memories
memory_store = load_memory()

# Remove older chunks belonging to this same document
memory_store["semantic"] = [
    record
    for record in memory_store["semantic"]
    if record.get("metadata", {}).get("document_id") != document_id
]

# Add the current version of the document's chunks
for index, chunk in enumerate(chunks):
    record = {
        "memory_id": str(uuid4()),
        "memory_type": "semantic",
        "content": chunk,
        "metadata": {
            "document_id": document_id,
            "filename": DOCUMENT_PATH.name,
            "chunk_index": index,
            "source_type": "document",
            "stored_at": stored_at
        },
        "created_at": stored_at
    }

    memory_store["semantic"].append(record)

# Save all persistent memories
save_memory(memory_store)

print("Document stored in semantic memory.")
print("Document ID:", document_id)
print("Chunks saved:", len(chunks))
print("Storage file:", MEMORY_FILE)

Document stored in semantic memory.
Document ID: 5e61c000e3431bbcff4cbb9a5b9d10cf75bc88dbec75f25eb18d0a2be22a4250
Chunks saved: 1
Storage file: C:\Users\vishakha.more\AI Assignment\ai_memory_data.json


### Retrieving Stored Document Chunks

This cell reloads persistent memory, selects the chunks belonging to the current document, and displays them in their original order.

- **`persistent_data = load_memory()`** – Loads the latest persistent memory records from the JSON file.
- **`persistent_data["semantic"]`** – Retrieves the semantic memory records.
- **`document_memories = [...]`** – Creates a list containing only semantic records associated with the current document.
- **`record.get("metadata", {})`** – Retrieves the metadata of each record.
- **`.get("document_id") == document_id`** – Checks whether the record belongs to the current document.
- **`document_memories.sort(...)`** – Sorts the selected records using their chunk indexes.
- **`key=lambda record: record["metadata"]["chunk_index"]`** – Uses each record's chunk index as the sorting key.
- **`len(chunks)`** – Returns the number of chunks generated from the original document.
- **`len(document_memories)`** – Counts the chunks retrieved from semantic memory.
- **`for record in document_memories`** – Iterates through the retrieved chunk records.
- **`metadata = record["metadata"]`** – Extracts the metadata dictionary from each record.
- **`metadata["chunk_index"]`** – Retrieves the chunk's position in the document.
- **`record["content"]`** – Retrieves the text stored in the chunk.
- **`print(...)`** – Displays the document filename, expected and retrieved chunk counts, chunk indexes, and contents.

This cell checks whether the stored chunks can be retrieved and ordered using their document identifier and chunk indexes.

In [46]:
persistent_data = load_memory()

document_memories = [
    record
    for record in persistent_data["semantic"]
    if record.get("metadata", {}).get("document_id") == document_id
]

document_memories.sort(
    key=lambda record: record["metadata"]["chunk_index"]
)

print("Document:", DOCUMENT_PATH.name)
print("Expected chunks:", len(chunks))
print("Retrieved chunks:", len(document_memories))

for record in document_memories:
    metadata = record["metadata"]

    print(f"\nChunk index: {metadata['chunk_index']}")
    print("Content:", record["content"])

Document: sample.txt
Expected chunks: 1
Retrieved chunks: 1

Chunk index: 0
Content: Company Leave Policy
Employees receive 20 days of annual leave. Leave requests require manager approval.


### Verifying Document Chunk Persistence

This cell reloads the saved memory data and verifies that the document's chunks can be recovered with their original content and order.

- **`reloaded_data = load_memory()`** – Loads the latest persistent memory data from the JSON file.
- **`reloaded_data["semantic"]`** – Accesses the stored semantic memory records.
- **`reloaded_chunks = [...]`** – Selects records associated with the current document identifier.
- **`record.get("metadata", {})`** – Retrieves the metadata associated with each record.
- **`.get("document_id") == document_id`** – Filters the records to include only chunks belonging to the current document.
- **`reloaded_chunks.sort(...)`** – Sorts the recovered chunks by their stored positions.
- **`key=lambda record: record["metadata"]["chunk_index"]`** – Uses the chunk index to restore the original order.
- **`len(reloaded_chunks) == len(chunks)`** – Checks whether the number of recovered chunks matches the number originally generated.
- **`assert ...`** – Raises an `AssertionError` if the expected number of chunks is not recovered.
- **`record["content"] for record in reloaded_chunks`** – Extracts the content of each recovered chunk.
- **`[...] == chunks`** – Checks whether the recovered chunk contents match the original chunk list in the same order.
- **`assert ...`** – Raises an error if the recovered chunk content or order differs from the original.
- **`MEMORY_FILE.exists()`** – Checks whether the persistent memory file exists.
- **`print(...)`** – Displays the verification status, document identifier, number of recovered chunks, content comparison result, and file existence.

Successful verification confirms that the chunks were saved, can be reloaded, and match the original chunk list in content and order.

In [47]:
reloaded_data = load_memory()

reloaded_chunks = [
    record
    for record in reloaded_data["semantic"]
    if record.get("metadata", {}).get("document_id") == document_id
]

reloaded_chunks.sort(
    key=lambda record: record["metadata"]["chunk_index"]
)

assert len(reloaded_chunks) == len(chunks), (
    f"Expected {len(chunks)} chunks, "
    f"but recovered {len(reloaded_chunks)}."
)

assert [
    record["content"] for record in reloaded_chunks
] == chunks, "Recovered chunk content does not match the original chunks."

print("Persistence verification passed.")
print("Document ID:", document_id)
print("Chunks recovered:", len(reloaded_chunks))
print("Chunk content verified: True")
print("File exists:", MEMORY_FILE.exists())

Persistence verification passed.
Document ID: 5e61c000e3431bbcff4cbb9a5b9d10cf75bc88dbec75f25eb18d0a2be22a4250
Chunks recovered: 1
Chunk content verified: True
File exists: True
